# 06.05 · Programmatic prompting with DSPy

**Chapter:** Prompt & Context Engineering · **Milestone:** M2

## Goals
- [ ] DSPy's idea: declare *what* (signatures) and let an optimizer find *how* (prompts, examples)
- [ ] Modules: `Predict`, `ChainOfThought`; evaluation with a metric
- [ ] Optimize a program with `BootstrapFewShot` and compare with your hand-written prompts from 06.01

## Resources
- [DSPy docs](https://dspy.ai/) · [DSPy tutorials](https://dspy.ai/tutorials/) — free
- [DSPy paper](https://arxiv.org/abs/2310.03714) — free

| Hand-written prompts | DSPy |
|---|---|
| you iterate on wording | you iterate on the program, the metric and the data |
| prompt tied to one model | re-compile for a new model |
| quick to start, no dependency | needs a metric and examples; abstractions to learn |

DSPy shines when you have a clear metric and a few dozen labelled examples. It's one option, not a requirement —
many production systems use plain prompts + evals.

In [ ]:
import json
import random
from pathlib import Path
from typing import Literal

import dspy

from aieng.llm import client

c, MODEL = client()
LOCAL = c.base_url.host == "localhost"
# DSPy talks to any OpenAI-compatible endpoint with an "openai/<model>" name + api_base: same provider config as aieng.llm
lm = dspy.LM(f"openai/{MODEL}", api_base=str(c.base_url), api_key=c.api_key, temperature=0,
             **({"reasoning_effort": "none"} if LOCAL else {}))
dspy.configure(lm=lm)

## 1. Data: same tickets as 06.01, split train / dev
DSPy examples mark which fields are inputs.

In [ ]:
tickets = json.loads(Path("tickets.json").read_text())
examples = [dspy.Example(ticket=t["text"], category=t["label"]).with_inputs("ticket") for t in tickets]
random.Random(0).shuffle(examples)
trainset, devset = examples[:12], examples[12:]
print(len(trainset), "train |", len(devset), "dev")

## 2. Signature + module
The signature is the spec: input/output fields, types, and a docstring as the task description. DSPy turns it into a prompt.

In [ ]:
class RouteTicket(dspy.Signature):
    """Route a customer support ticket to the right team."""

    ticket: str = dspy.InputField()
    category: Literal["billing", "bug", "feature_request", "account", "other"] = dspy.OutputField()


predict = dspy.Predict(RouteTicket)
print(predict(ticket="Why was I billed twice?").category)
dspy.inspect_history(n=1)  # see the prompt DSPy actually sent

## 3. Metric + evaluation

In [ ]:
def exact_match(example, pred, trace=None):
    return example.category == pred.category


evaluate = dspy.Evaluate(devset=devset, metric=exact_match, num_threads=4, display_progress=False)
scores = {"Predict (zero-shot)": evaluate(predict).score}
cot = dspy.ChainOfThought(RouteTicket)  # adds a `reasoning` output field before the answer
scores["ChainOfThought (zero-shot)"] = evaluate(cot).score
print(scores)

## 4. Optimize
`BootstrapFewShot` runs the program on training examples, keeps the traces that pass the metric, and inserts them as few-shot
demonstrations. Other optimizers: `MIPROv2` (also rewrites instructions), `GEPA` (reflective prompt evolution),
`BootstrapFinetune` (distils into weights).

In [ ]:
optimizer = dspy.BootstrapFewShot(metric=exact_match, max_bootstrapped_demos=4, max_labeled_demos=4)
compiled = optimizer.compile(dspy.Predict(RouteTicket), trainset=trainset)
scores["Predict + BootstrapFewShot"] = evaluate(compiled).score
for name, s in scores.items():
    print(f"{name:30} {s:.1f}%")

In [ ]:
compiled(ticket="My invoice shows the wrong company name.")
dspy.inspect_history(n=1)  # the optimized prompt, with demonstrations

from aieng import DATA

compiled.save(str(DATA / "dspy_route_ticket.json"))  # the "compiled prompt" is data: version it like prompts (06.03)

Caveat: with 18 dev examples, one ticket ≈ 5.6 points — compare methods with care (02.03). DSPy's value grows with harder
tasks, multi-step programs (retrieve → reason → answer) and larger dev sets.

## Exercises
1. Compare the best DSPy score with your best hand-written prompt from 06.01 **on the same dev split**.
2. Try `dspy.MIPROv2(metric=exact_match, auto="light")`. Does instruction optimization beat few-shot bootstrapping here?
3. Re-compile the same program for a different provider (Gemini/Groq). Does the compiled program transfer, or does it need re-optimizing?
4. Build a 2-step program: `ChainOfThought("ticket -> category")` then `Predict("ticket, category -> reply")` drafting a first response.

In [ ]:
# your code here